# CLIP event-to-frame diagnostic

This notebook enriches the existing 900-row `events.csv` in place. For each event, OpenAI CLIP ViT-L/14 compares the event text with the row's four shuffled input frames.

The stored matrix is **not** CLIP's normal scaled output. It is the raw cosine matrix `L2(text_features) @ L2(image_features).T`; the learned `exp(logit_scale)`, temperature scaling, and softmax are all omitted.

## Setup

### Key assumptions

- Matrix rows follow the JSON `Events` order.
- Matrix columns follow `Input_1`, `Input_2`, `Input_3`, `Input_4` from `train.csv`, never directory filename sorting.
- `CLIP_SelectedFrames` is an independent 1-based argmax per event. Duplicate frame selections are allowed.
- Event text is passed to CLIP unchanged; no `a photo of` prompt is added.
- This is frozen-model inference only. It does not train or fine-tune CLIP.

In [1]:
import os
from pathlib import Path

DRIVE_ROOT = Path(os.environ.get('SNU_DRIVE_ROOT', '/content/drive/MyDrive/snu ai'))
FULL_DATA_BUNDLE = DRIVE_ROOT / 'snu_colab_full_train_bundle.tar'
WORKDIR = Path(os.environ.get('SNU_CLIP_WORKDIR', '/content/snu_ai_clip'))

OUTPUT_DIR = DRIVE_ROOT / 'preprocessing/qwen3vl_event_chunks_900'
EVENTS_CSV = OUTPUT_DIR / 'events.csv'
BACKUP_CSV = OUTPUT_DIR / 'events_before_clip.csv'
METADATA_JSON = OUTPUT_DIR / 'clip_event_alignment_metadata.json'
RUN_LOG = OUTPUT_DIR / 'clip_event_alignment.log'

FULL_DATA_SHA256 = '892E237C27C6835B2A1F69C45564B26B7148F0F71FB70761234401B5D746C1D2'
EMBEDDED_CLIP_CODE_SHA256 = 'A50E2ED7AC2C27933EB6FF96A4971212EF71EA43C3EA452B569DF375D23701F7'
EMBEDDED_CLIP_CODE_ZLIB_BASE64 = 'eNrNPWtz2ziS3/UrsNy6GjIj05bjZLLa1dRmnWTGd3lVHnNX5XOxaAmyOKFIHknZ8fr8368fAPEgJSszW7ebD2OKABrdjUa/0OAs63ItkmS5aTe1TBKRrauybkVaFGWbtllZNKORfldfVWndSP173lzrx1XarPLsUv/8tSkL/bxO25V+Lhv91Kw2bZbrX222lqMlYrJI23Sep00jG41K94p7VAAPptKt7xE8NbS3VVZc6ffPi9uxOGtlnV7mciw+yv/ZyGIuR6PRm3cvXr5O3j5/81LMRPBL9ung9eHkJBid/vzy9D/evzt7+yn5/OE1tIUjAf+CVdtWzfTwsKxkkWbV5jLP5nH6d2CXXFzJuJDt4TzPqsN1uZB5cxjwqMtn83n6eLk4maTyaP6nP12mP8hnf3oySRfLyQ+L46c/zBeXz07mi2fPLuXT5Q/zY3kk5+nkh8c/pEePnz1+quEQfgeTk7hqg1FkI/nx5+fHT54iDb93rmB09ub5Ty+T03evP795+xFJD86KatMmk2As1OOxeXxsHk+CaPTx5euXp59evkhefQCmflRgELHT12fvk48yl/NWLl7V6Vo2wej1u5/OPiVvnn/6cPZfft/X5VXWvknbOvsajD68/Pj59Scbq+GZxmIAZAQr/ddOdEKQkL/LYvap3shoRK/Ey2tZtM/z7KpYw8OH8mZKHG/SdZXLJFtMRdPW/AraUXjMG4ljm6loN9D3HN6ORRzHF9SWrdMrmaCYdh1QSMfC/y93h12U1DC5WGTzlkHBfy4A/YVcCtpwCey8JozEwY/dHozfIjOrFJAiKPSyBh51HZ7XVxsk7D21sCzjv4Vs5nVW4daembcka6flGsZKMU+pWVEpbrJ2JWQ6XynefNeIZbmpRYYSIJa0rGLT4OYLXICtXFewA1G1HCxhvwhcZDEvoa8UOa51E5sh0cj8l+mJ08UCiSdCwuDggDE6AJ6BCMKGlzNmZw3bO6vlQi3wDghtnWbF7wFAy3tQl2X7myB01AKoctMCBxmZ7r0BaS3ZMt3k7extWUjzdiXzahaotka0pdhUIPC4DDajRNqW62ye5vktrJiocpAZNd1OOteyTXH/HKAyd0m18dkJ4zKdf9lUfW7vDYB06kEBEhaYUUaB7xw7X8n5l6rMivZgU+fWeFfR7wujWaWgbofBsCreb9EtkKgj9lt5fBsGh/MSFRHbm4R4k8xhX8pD20pEDy3uFmzWWV2X9W+XxJKURpoL2PJN1iCi4kWdXUvBkGHfV5lciHQJVllcyzpb4s9FeVPkZboIvgVt3oSXaTtfHTTZ36WPNdDTR/rIx/gINDvapkY8PTmcHD87RHvKbohsyWaJn95/Fr+Asdlnx7Tya+vgZHDpcHgyOf4WCm/K+gtw8/eSt6lQPTwTp0BNVcuqLueyIYXtTrCTPBi4lEjcMp23KCgD5J3sR9wCR1pEzVdlBhjNwmAJotA+Jl+DHidPtTzbpHZtNtq1BDtT6Lltw6lsaVKU9TrNYWkWiTbp4XWab9iwk32Fv1MbWiCC+FfYHtwvbqo8a8Oog1jLdJGAegtxK0/JqhMYNvs57AK06BdjQY+uhb+44JnIvOL4GD3MMEDOAmrlAtZnFmza5cEzkKcreFvImxws5ywIIpGCEc5yZf0Z4XRBHgCgE7+AmT7QixB7RV2vbKk6xrD58kVBtjtrBG7oqWO8wU42UvyCZL/E7Rsug7elOP34C0gXzQQ+QAFO0h2ifh9EFiLEOaQ47M0Vje0GzUfc/0mqXTF0h5rQ8rOQw8xcXm+y4f5LdrvQLttvH/EfWL0MjBm4ddgL/LLLssyBV2irx6MdS9ZzEvWqEWaJIWys3iDuANhIhiFBiyn7CYnyrmbiLjhboMB/VDKJzzRvE9zTkHVGm9WM8GDEi2y5lDXJs48XTwrL7gKx5Ka/zm+4q/b/5mW+WRfAtLsGIiu5CF1QES6+tS42T/hNjyfdAnosoffDHHnkhCguX/QwF47NFR+zHleow15M4cm28oSaI70feN7LWwgnkCxgxDmSdhGDiGVVGE0FvIKdVNNfgGsYdq9RzEEpWHAi8YeZ9Q67RjsQD16AXIP710pxtuA9282Du1mvXSNlAdCBokbyDgCE4TFkQtzNOd22OWDMOYc1SJMKbXA+szkMrl2kxUor6XHH1lpF2Vqh2W5NFbxcV+2tQ68SZSbYAmtwgE4dDx5ShMM8NXOgWGjAtmrUE6BlDLsepr1bUuCIteTxFazDQHfA34zYV4mzZkHEYUR6iYaQfZ5OJLZjz7sRt/KQKe3WsNu2Fy6v9fg/bAHQEeOAeJAi3Rc38xrdLxK+YRra+taFt5BgaW1tjOFOjAapMQQpbWyRI7/OZcWJrvjfP757+4LAEEZomqH5QbTPCrJKQi0IQukjzksD4Py9kIG2adoU2eaSwIYrEgAJ+03EX1hduL0ifH2yr7CsN00rMPhACZmgL3mCLgzou2YHrwHTtLgNPWxp/jE7XApHeqU3PQGkN7inPKz3RVjhCgIOExQHjCtOJ0kzkDO3A/NOGMgpCAeFlfrsxtZwwqSEOqAOJcZ5EYeWTjo0+/ucLU5fN3bKltoRC8dSetkVtS6+P+RA06ZM43uODzQHPcAMNj1KHsmBzZoE3c0wunCpW7ow3enMSr6CsW/L9hUqVF7QXkeitrPFhEfYRP5STr1ElP73vUCvgh164CN57ZFDmYNn1ANi3rgGMU4r8N0XLr4989gnp0N51j2N+52UyM30Q78LC9xMaYBes7VeM+t5LMQfKYibCkCyrOU5xEsH+OKiB0LpwhnGMFoxRu5MkSdrSjI812G0O8AgX3HYpBpPWu9VFWr0fXF3ThVlWNnIBLORCWcgQ4zbk6WkBnBt6OSAueS8pCgB/jIFQRB84NmddCZFc+WmpUzndw1o3rQu5AI0eBVSj6SZpxCQxTB+pOIWPgIp6/kqLop4uSnmKpGCMV73S7nYNq6wN0173Ckpl6CYYuUQAq9Ftp4dTCIrWnoIkNtrGJJaAxexv3oTxKDHiqYqGxkejcVEx32NOhpIOImcoLHTq8LcWtNZgOE/+Z1Z0aooDEQMfKO0hb1s94dNXgCOsNkpw4gOxzFqf7tP3KzSSp5PLrD1ZJcHTUlrGip4KNvDVXotBQER5yyBcxBbMG0nF654Ir6cP4hAA01I37AFAgl3UILtt06/hsjeSRS3JYXKkT4LSCi3BLEGDQah1Ida58AdCFgx6QQ7Ga0ScEqffZ07vQzfsDc6AUe7SKcZuSvRfCkFLGLWZtdS0YjEgGWvyf7BIl/RCqPDwVhGjJdluW9xnzIDmnMeOlUgvqe+Hb3ppi3ZQjHpSUPSXbZpnlyDxCRX2eVUkFQSyUB6R57bS/w4E4+fxDatvDbHT55uH3F8NDBicvzMXtynJwPYqqxaOK82LBS0LOJ/SRxhs7FU2hin12mW43JhLKRHIVbWj4bUKYGQOSxUWDZx1xySN3XsCN4azNyzsUCRmoytKQ6gX5fC0olYttwmjzVGncW7k1PfJkmGyZOpm3OzjT94fQuhBikTO5v50MDFvpG1yc3pNHBSLpfZPIOVMKlptpubOicU2OgMIcctZlxiUeM3cU6aG9XCqEQQvul0/AuFlngH9v35GWh0jZ8FDPRhDcjkt5zLA323WUviAs8CDoXR9x7iYjtnBojBtCb6yusvMGXIPxo6ccLlAm2RlF+sA6ist74evN4iW/u0qpH1y+AzuVpdvj4vwYApdqjzvA4mGGlvgoHEoNdDI9pbG0fgSaRcUqzssTeyRxVnifukfZBNS8tjyBsmjMMwOs1waeQJnQhTokFP61vcyd7qoXgkzQYk6GsYxFUbz8vqFqa3g2wqjKCG43CAsg78wIxxLel4z1/lB5dg1AuNlVeCGTXgUTOyYt4zahoOddmCfAB9lK21DTnDyC/PO1ic7sI9QsqION0dAlkBrz5JaHGz7cFL7Bpsk3sFZ6e8c5eHmWikZ17CLCA08xKd91b2JEajT6d0v3l35ClsbMn66iUtAp7OO9aks8XgC2GMa6wxaH6byl4CBPRZI1sAo+mH1QJrAn/I3CIvdYvR8GR/jhw4fCrA+ewPOHMApjm4vG1lM7vjSe4PgnuEp6YkIHf3DhTm7agf8mlFjJtV7VFdutPjeqeiT1+8HYgEl0GouXSnHu4Pn4yV3k5clKM4joN+RLXMN82KFe+WGKg72tFyT1m8HiCwan3oipcz9XcgJGxrma4H5qclhr0HIcgsfAye2NOjIz9Mw10LtFZl0ch+KE4nRNxIorCBiKtcSDThJ5OnOxWxs5BbNlw/pJ7uGv/wbrT/XQJXvvSTC5oc0k4J7JKECQv7UDiOByFWMvqjOCKSt7Hk+KhP0ZraRJBeBpQDY5Ak7cHNZTDEcTUbzkQhMw0ZZky3X4/6MTqXByTgfV+B4M3QvQw7zJUwkRgGqusBd8WsyFEQ9dnRLR5tCsOW7/250IC7bwZUBOVb1T6Whsm9PqTvYOHRAs1InGMI3Jfs6HauUW+jaZGhY1RchIFT0l7WbLUpvpCy1GzKYIZE0RJSM+nB2TPxSEyOjk/Uny1ia+U9aOz2XnrBsmIjt3ZC5OObGnBiVKKtPS2+fs8J3wcGFHzKsAdzHbpuIHqw1wfCpMnRbiphnCtHu7vvMAPD2UDH/7T4cEgL9ejR8TQ+Wt4fBnuCu/OE3gUj3mR/G7AHg6C22Iihf9HOHriX/ilM+6dQ625/EDk/mWxiS9rwlN/x48HRg1JI2naL0/OXvSR2yNu1o0IdO4IuB85eSlA4sgNMqY6gzwmN0abIs+JLd67sxHXfgMTHn58fYJipz8b+jJoOky3oSBm3e/Sbba9rc1WMYLzUocMw489S9Dv311d7Z2Bfn0z3IXhwdQIdyoETOLQqyxSUqy5HW2Kdmpq3ibccHwSfVrJz6VE1Y4gKfnuaYYoXzUkLHQr5FQOnelPEfTAqvjEc8A4j8xTjxskRWBmFzYB7vAxOt9KDefKv83tcZnD16ZCkgA2OgO8bsPNmm7rLSIagyaWsQuqsygL2Cszxxz8uOO+N+MakhxMSa5RnA2D3iMY1vz+m106WwJInNUFb7pce2BbfczbGx3s7RUOb0x6zLdJXxe/JaZ5VZ5isfJG2KThhzH9KxSVZkbVJEjYyX47dcvYul0z167qRkvsg/muTp3dLERBS7B6CUmrbehUNde4Aoy/rvhlZ+IK3o9B1k6p2NRo4RD4WkQ0DvGLwstaGbNDZX002vTv3IVOHW/j92WudJSE+jvnPu6oZWSkFcsP9ic8JuDljI++VRrPvyjnTtGGavTPvq0uAqKeKQf6XiTlcoQERSFcBogpu/oef/ubpdsWPISaHALvLxwKIZXaFR2V87yWhmuOqlvOsAb1O/nWuDsuonDPBozeTI7YPyt6DyZE1KFgrufZdI169h3gSUyebVoLf8er942PxOr2V9Vtc8arMs/nt8DlZp5vMzGgsuupRwzJCUx9c+ULBZ25q0ADIP8xMFerOc8vPRbOpKioEcxKICz5cvTMwu1q5PwpkKBWaYMY8x9q4Wwz88gxcE7QjJtHMq5ncyOxq1TaKM1NxCu9R/yp4r8EKpXXHUN0bL3Qgp8fiZoUmy+JwiqduLSZtwCO4pQWIFbDTNM9R4zH/2jJUfIjESgLAm3KT4wHtvKxV5nsOFs2CDerw5xREjJPYjFJCbADpZaDXWbNJc5LVSczYxtRD5bwAFBVe8DBKLNm5OkbdGqFDOoC+yelsjefh341bTmYVpHD72JzAdlREDtxuTgUaBsWGhRBnzzd1I2evUnDWeWRXwucyAMTKkrzJ053pU7fOI3DS095K6xM6Xu0rMMp3zsT3Qf+Avs9mwO7O2Rj3++PnoDcoaIRjLddYSITy9mfP1VoGhLcuqkyrkOqle2hGUY8abfFskrU2o5Ml3HCML1+d6BTWt5wbuadQVsOuc6jBwyayPa7mdEqRSbVu+Y86xh1K2SOV/jtWmo75Iu3TZFR00pZfZAHRSK37f6T3n/Trf0T2n63IAU6rtpr6c032ZEyO5Pssz8sbv+pN5VNYKOebRYqJ6O40M4x2I/BcnH5+8ZwucpC3ziXCyluHN6clgFHqOkuvirJps7kymgt5nVG9I8/NPyG8AhymR90hf1Pm1+DNWj7hbJ/zzL5gjYfes1wNNlXO3Zy+p21djiCJ7/lr6BZ3VmhkJZfDAaqw7oMYMOM/Y/Fr1rK6sybSOn4/B8L2HSxEY3md5kphM+vxJhmE0Q2vP0Sy600epygtSbsEv4FL+bcMAJ3+DV0vwcFdrdP6S78n+MmJ0ooJ42ARFKxADbuVHx67x8xtvd3G/j7D0hrNFsViR4vRrRBdsxf2L5t63rnVwfXQx4bT1m+e0nphai7ID+aXqrTBeqPvBSV8L0i1uE5gTw9xxTExFeOhJsarhrobhiSvMR1VqyPzgssquf6lMUXmC45dUNz8eCZ06tw8LvAiUcarTr7c4C0h+/4tKljH3QgMK4KpxRez+QKI65bLHJtpR1gtxWatC0KgVT1Z7VVWJGu5LutbaHbzZ8GiLqsEkwUu3M6xUNDwmMJoQYeumC6Feoc05oaexmwgceet68z7PfbML88KbDOrF6r1GYtHjxyk9M7GghuzqBijcTdupzQVnRNsy1KjEufVoIhqLKrsKxh2ri2i+kFgPpb8SQV4zEBnE/vc1x4zc36i4+slaBwFSLtzxiqLnsdYbpxc5uX8C97e6tSHm/SkeI+FPyvUFRW6zhl6pwpWfZ7Si5YGCG1Mrdpld7PoylS/iM/xhi0eYhw1ES5fxb9hRgoajvyG2cxdRRd9madVs20BxYFeYE/q6G4ijcL6JXu2R/bWI3Gxo/lov4NkGiI0vuLOmuD+8M6h5n7w8JjVbwLUJI0EK7eY3RmkD6naShE+FhN58DSaxpPl/TefITsx6hyWzFtYrsI8irAEzJRRKQHBakxlIejRsg10uU3tP7Y7v8sm7NDzu3W3v/vxOyHxXGZ5SJebEOsI2Gkm3LLl+Qf1d3d8Vy5JbWMb0pAaoK8ZUHm+YktIKSoLfIT32TYF3vvhdOMW9fD/pAQQp5Cx/pfc/Tp3injuud8G8tO/bRvwNw8SOjmlG4cjnY7TxY18ja3bFd7VXMCjqyb39s7Iy2/+hrI8u0SsV8tEL/gZ7yvE7bpSTiVLTFfsxbeFb/q3hR+6KUxcsW8K/ye9oJvCNuEz2gDO3V1YyjptUqoUnwV8acBKMDJkPrDmcoNwuJFu+NLVxdFwBRufMw2tJl65cspUq/QWjbvvwP3rL9PA0tB9ssVmXYWKqrHgVZFFg6FU2syzjGMuTlEX7ezYSlOZcoHgv4tgL+ZadbrWBfbuHvwiA2OHHrb6xFOs+kfDd9cvh6hyCi1I1PJ0fQkxNCOMqc/QLq0AXQ2C693TIzy0N2vVNigFodpX8is/dbqApca6ECIb/F5KOHiLe9BO+ndfuy79+6/j/iUNy3gOXCrhBv4QjH2jnRbBFeipWz7MFm8mms2aLCbgFuu7fdaFYhf5yIXBrgwAQdYjkKHeWBrvXCjhixNHF86xO7sZO64SfEIzoGAIrmhflJJPEtWtzRyPPOm2DN+37Ob3LpMMIuDfZOtjQOHhAyjgd43UF43AwUMm6oS9WiPr+rNnM4xr011rWUG38grAYTSJaRUye9EUTCkuEVtRu1LzSXRvJMir9FKLqv0efbG683e8tZu6txnVJQKOsixZMfzC/Ih3u8j5fW4jNXVQ/N6e48IB2bto5L447ygCCTwBsKF58b2YRPjWAFTXb2Zbr3L1iBgPIOEcNXItxmznVSSe1j0NoG5Jt4J6uKu0+oKA8hJaY6MLLNeaeBQmKF8oT+fsr6nLQ87FIY77LsxLcn/VRSJzdagrG+zEF+DSJT4UAn2Rb6Db+fDH1i70FWm0Uk2oKRyEMPBZNm+4Re4QhM5/Na/sE3FLAGe2BPKGtT5wApykAz3kFT1k/UuDlGbGNrqvXAj363MXHsgYJkfE3F6MXN/7NPp9bFNnO1uOMbMyT9gvmNKutUZaJb2BUpZTTxdbPVg12j34jZ2+csTfyGswHRBia5z6DgbWepvrlypHxfYXj3pCzxnDHBBXnJtv9OjaH8xZ+Xe+xF92X1PrfxCqOwyjoMW7s6anIQmyZ5k9NI33iadtl+JcOnSSbk8idPd9KPBycntQ4H3FaRsFRmL5m4JNbL3Box98ZS7kcvJAfTGOXHQ9zH0JIw0c9qdRZKyPQ9EpAe9NY9A0kJi+RWedNvAH5mw0rTfbJlM7JuGCPDqXQEWoKH/oMjGa0G2fK9KybXFmrERNf/FmbEsGXvi3qEEHyHyIgi+Vbvfl3E8PGJN+YWC5JTfmHv92ePoePIKzRl+Mhg4TxsIcZVxW1jli7xgjwRS9dwjbMcucxo7dhq1HXX6rc3bmN/oHaH67fVjWtXM211qbfmarqiHmqduMPSVzOHkl24TpTkyXkN/Yie/upunMAhVzG58HdMW4jy3uW/pqtkVbwjrudXfWvgqutc5sSHN5APX11sikEVXVP4kZ2ipHHFTGihJxEd6BPubjV4wIQNYssVefRMXq+6/ePQbOfblN+iNLnJRT393oPF0VGSGIH32YOL++ulC4JKhP2Ln54yUZ4tndQJB0r7++QI0WMdDCdtZqsRPW97Ym682HJ9IzVU9x5+2Te+uwd3Y3tFfs1PUy4HOKOyPZbrPLm9md+/veZzDQM8jy3fS0q7rcXK1AHU97kjy789+4CHqGWlHivb13JVnDVD9dgP7Z1t2QNb0XVVZAOKp2I2WGbBIfPp+ytAKWwMHSyPSLAkd3kFy9wAVaOM2O+3q9gGroWNg9ZnDlbtvnQYxaH7zRNHBNTWn3LfCGjkndHr2zUEcFbz1v7N8gGKxBZ+a/27Tvlm+I4+QDTf1i876vOROPj7eUnY92kSkOD2fi2K2qNgJAXx7iz7WGex5VUeHKu3dvplYxN+UwdCaR828799E3nzv5OYCBIyXPV7E+TqrdgdEWyfElZmgfa6urv8YtE/bi8L6YcedUgUpI3xbwfC7TODKfeevA6Ip14ynGlPxt7IMXq/Hbq8+dGu+eP2hAdwVEmxzclM16zbnmnRnLQRd1y1eJxm6s3OVERsO6ZOxH4Iiupe++5TjX+tYNflyGshhswa2WeAF+/XwVRvpYKsbP5KijnMiJKdxyDL6ECUFnoG9eB72AGLGHHpbIpE2CUc7X0I6ezWpQZUe36qazLz90sdH9AHLQufcAxPX3h2c17r8eYN4MjyDmQec7t+AyW9A1a7vusucseCogcB1njcA2d9ofor6DPX3Qz/YHqu9LT0U4rO/7FzQMI4Zucw0Psi+njIbu0fVbvPvR7C1pAq1AoOugPuqUWKKcgORimsa82WMMV3sv+jVD93ZmJltneQoq4ba/+mitN3mKG+H1sfvZp0j8VcA7L/MZf/KFwc6klvV+6DFe5bJdp18f6KUSfCqLpT/PV9YLWQfDXU1GiVKYzsfs/JXi1FRWFggb86mYLMSAmD+kRCl80gZ/Fovuk50qvU8VgXIRDHOdeKn84D7fkW5OmJRFIteXcoHWGHF2Xedxf1h5SVchFhwd4RAsGHEDEH+JhpxtGDj4fpgcsq54tamvQDyHIZg+4L75znin8YasuDeNKYPb4QD6zrmeYKc/GDiueq+QjgvxrKoM5e3KutfVZttVtelzjP5vBFM7ZndTF8wkJ9DGxXJe+MhjWIACycupR3S+IwqOihq6XjpyMEmCQRoePXL9C9tOsj1XVUyUH1Z1S6aPMp+d0tfHxcasRnZJ4sBRvZMAHHcGfUuU+JnOdxfizkxwz17unUvH+XeoU767uOcvBqJTZwd4vd7sFGB/ejogLSA6BUKHJ3eKfirYavrR7DJ4o7AHoXDooms9I7yaTCY3Seg6UpJg8jtJ1OUhzoSP/g/ohsjh'
MODEL_NAME = 'ViT-L/14'
CHECKPOINT_URL = (
    'https://openaipublic.azureedge.net/clip/models/'
    'b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/'
    'ViT-L-14.pt'
)
CHECKPOINT_SHA256 = 'b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836'
LOCAL_CLIP_CHECKPOINT = Path('/content/clip_model_cache/ViT-L-14.pt')
DRIVE_CLIP_CHECKPOINT = DRIVE_ROOT / 'models/openai_clip/ViT-L-14.pt'
SAMPLE_COUNT = 900
IMAGE_BATCH_SIZE = 0  # auto: 40 GB A100 starts at 256, with OOM backoff
TEXT_BATCH_SIZE = 512
IMAGE_WORKERS = 0  # auto: up to 8 CPU preprocessing workers
PREFETCH_FACTOR = 4
DTYPE = 'float16'

TRAIN_CSV = WORKDIR / 'snuaichallenge_data/train.csv'
IMAGE_ROOT = WORKDIR / 'snuaichallenge_data/train'
print('events:', EVENTS_CSV)
print('in-place output:', EVENTS_CSV)
print('one-time backup:', BACKUP_CSV)
print('CLIP execution code: embedded in this notebook')
print('embedded CLIP code SHA-256:', EMBEDDED_CLIP_CODE_SHA256)

events: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900/events.csv
in-place output: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900/events.csv
one-time backup: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900/events_before_clip.csv
CLIP execution code: embedded in this notebook
embedded CLIP code SHA-256: A50E2ED7AC2C27933EB6FF96A4971212EF71EA43C3EA452B569DF375D23701F7


In [2]:
import base64
import hashlib
import shutil
import subprocess
import sys
import zlib

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'openai-clip==1.0.1', 'requests>=2.31', 'pillow>=10', 'pandas>=2',
], check=True)

def sha256(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest().upper()

for path in (FULL_DATA_BUNDLE, EVENTS_CSV):
    if not path.is_file():
        raise FileNotFoundError(path)

full_data_hash = sha256(FULL_DATA_BUNDLE)
assert full_data_hash == FULL_DATA_SHA256, (full_data_hash, FULL_DATA_SHA256)

WORKDIR.mkdir(parents=True, exist_ok=True)
shutil.rmtree(WORKDIR / 'preprocessing', ignore_errors=True)
embedded_code = zlib.decompress(base64.b64decode(EMBEDDED_CLIP_CODE_ZLIB_BASE64))
embedded_hash = hashlib.sha256(embedded_code).hexdigest().upper()
assert embedded_hash == EMBEDDED_CLIP_CODE_SHA256, (
    embedded_hash, EMBEDDED_CLIP_CODE_SHA256
)
package_dir = WORKDIR / 'preprocessing'
package_dir.mkdir(parents=True, exist_ok=True)
(package_dir / '__init__.py').write_text('', encoding='utf-8')
(package_dir / 'clip_event_alignment.py').write_bytes(embedded_code)
sys.path.insert(0, str(WORKDIR))
print('full-data bundle verified; embedded CLIP code installed')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
full-data bundle verified; embedded CLIP code installed


## Steps

### 1. Validate the event file and extract only the 900 sampled image groups

The existing full-train archive is already on Drive. This cell scans it but extracts only `train.csv` and the 3,600 images belonging to the 900 event rows.

In [3]:
import csv
import json
import tarfile
from collections import Counter
from pathlib import PurePosixPath

with EVENTS_CSV.open('r', encoding='utf-8', newline='') as file:
    event_reader = csv.DictReader(file)
    event_rows = list(event_reader)
assert event_reader.fieldnames[:3] == ['Id', 'Sentence', 'Events']
assert len(event_rows) == SAMPLE_COUNT
assert len({row['Id'] for row in event_rows}) == SAMPLE_COUNT
event_ids = {row['Id'] for row in event_rows}
event_lists = [json.loads(row['Events']) for row in event_rows]
assert all(1 <= len(events) <= 4 for events in event_lists)

if not TRAIN_CSV.is_file():
    with tarfile.open(FULL_DATA_BUNDLE) as archive:
        member = archive.getmember('snuaichallenge_data/train.csv')
        archive.extract(member, path=WORKDIR, filter='data')

with TRAIN_CSV.open('r', encoding='utf-8-sig', newline='') as file:
    train_by_id = {row['Id']: row for row in csv.DictReader(file)}
missing_train_ids = event_ids.difference(train_by_id)
assert not missing_train_ids, sorted(missing_train_ids)[:10]

image_columns = ['Input_1', 'Input_2', 'Input_3', 'Input_4']
required_images = [
    IMAGE_ROOT / sample_id / train_by_id[sample_id][column]
    for sample_id in event_ids
    for column in image_columns
]
missing_images = [path for path in required_images if not path.is_file()]
if missing_images:
    print(f'extracting sampled image groups; missing files before extraction: {len(missing_images)}')
    selected_members = []
    with tarfile.open(FULL_DATA_BUNDLE) as archive:
        for member in archive.getmembers():
            parts = PurePosixPath(member.name).parts
            if (
                len(parts) >= 3
                and parts[0] == 'snuaichallenge_data'
                and parts[1] == 'train'
                and parts[2] in event_ids
            ):
                selected_members.append(member)
        archive.extractall(path=WORKDIR, members=selected_members, filter='data')

missing_images = [path for path in required_images if not path.is_file()]
assert not missing_images, [str(path) for path in missing_images[:10]]
print('event rows:', len(event_rows))
print('event-count distribution:', dict(sorted(Counter(map(len, event_lists)).items())))
print('validated sampled images:', len(required_images))

event rows: 900
event-count distribution: {1: 154, 2: 447, 3: 211, 4: 88}
validated sampled images: 3600


### 2. Inspect inference GPU

This is not a training gate. It reports the GPU, resolves throughput settings, and fails early if CUDA is unavailable. A 40 GB A100 starts at image batch 256 and halves it only if CUDA reports OOM.

In [4]:
import torch
from preprocessing.clip_event_alignment import (
    _auto_image_batch_size,
    _auto_image_workers,
)

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before running CLIP inference.')
properties = torch.cuda.get_device_properties(0)
total_vram_gib = properties.total_memory / 1024**3
resolved_image_batch = IMAGE_BATCH_SIZE or _auto_image_batch_size(total_vram_gib)
resolved_image_workers = IMAGE_WORKERS or _auto_image_workers()
print('GPU:', properties.name)
print(f'VRAM: {total_vram_gib:.2f} GiB')
print('dtype:', DTYPE)
print('resolved image batch size:', resolved_image_batch)
print('text batch size:', TEXT_BATCH_SIZE)
print('resolved image workers:', resolved_image_workers)
print('prefetch factor:', PREFETCH_FACTOR)
print('pinned memory: True; non-blocking GPU copies: True')

GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GiB
dtype: float16
resolved image batch size: 256
text batch size: 512
resolved image workers: 8
prefetch factor: 4
pinned memory: True; non-blocking GPU copies: True


### 3. Run CLIP and update `events.csv`

The script computes all features first and only then atomically replaces `events.csv`. On its first successful in-place run it also preserves the original three-column file as `events_before_clip.csv`.

In [5]:
from collections import deque
from datetime import datetime, timezone

run_command = [
    sys.executable, '-m', 'preprocessing.clip_event_alignment',
    '--events-csv', str(EVENTS_CSV),
    '--train-csv', str(TRAIN_CSV),
    '--image-root', str(IMAGE_ROOT),
    '--output-csv', str(EVENTS_CSV),
    '--metadata-json', str(METADATA_JSON),
    '--backup-csv', str(BACKUP_CSV),
    '--model-name', MODEL_NAME,
    '--checkpoint-url', CHECKPOINT_URL,
    '--checkpoint-sha256', CHECKPOINT_SHA256,
    '--checkpoint-path', str(LOCAL_CLIP_CHECKPOINT),
    '--checkpoint-mirror', str(DRIVE_CLIP_CHECKPOINT),
    '--image-batch-size', str(IMAGE_BATCH_SIZE),
    '--text-batch-size', str(TEXT_BATCH_SIZE),
    '--image-workers', str(IMAGE_WORKERS),
    '--prefetch-factor', str(PREFETCH_FACTOR),
    '--dtype', DTYPE,
]

tail = deque(maxlen=120)
with RUN_LOG.open('a', encoding='utf-8', buffering=1) as log_file:
    marker = f"\n===== run {datetime.now(timezone.utc).isoformat()} =====\n"
    print(marker, end='')
    log_file.write(marker)
    process = subprocess.Popen(
        run_command,
        cwd=WORKDIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        errors='replace',
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        tail.append(line)
    return_code = process.wait()

if return_code != 0:
    print('\n===== CLIP ERROR TAIL =====')
    print(''.join(tail), end='')
    raise RuntimeError(f'CLIP alignment failed with exit code {return_code}; inspect {RUN_LOG}')
print('CLIP alignment completed successfully.')


===== run 2026-07-16T09:08:06.760326+00:00 =====
Using verified local OpenAI CLIP checkpoint: /content/clip_model_cache/ViT-L-14.pt
rows=900 events=2033 images=3600
CLIP=OpenAI ViT-L/14 checkpoint=/content/clip_model_cache/ViT-L-14.pt dtype=float16 context_length=77 truncated_events=0
throughput: image_batch_size=256 text_batch_size=512 image_workers=8 prefetch_factor=4 pinned_memory=True
image batches: 1/15 images_per_second=54.1
image batches: 10/15 images_per_second=330.3
image batches: 15/15 images_per_second=356.2
text batches: 1/4
text batches: 4/4
Updated /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900/events.csv with 900 rows and 2033 event-frame selections in 11.0s
Metadata: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900/clip_event_alignment_metadata.json
CLIP alignment completed successfully.


## Checks

### 4. Validate the enriched CSV

This checks matrix shape, cosine range, exact event-to-selection lengths, and that every selected frame equals the independent row-wise argmax.

In [6]:
import pandas as pd

with EVENTS_CSV.open('r', encoding='utf-8', newline='') as file:
    result_reader = csv.DictReader(file)
    result_rows = list(result_reader)
expected_columns = [
    'Id', 'Sentence', 'Events', 'CLIP_SelectedFrames', 'CLIP_LogitMatrix'
]
assert result_reader.fieldnames == expected_columns, result_reader.fieldnames
assert len(result_rows) == SAMPLE_COUNT
assert len({row['Id'] for row in result_rows}) == SAMPLE_COUNT

all_scores = []
selection_count = 0
for row in result_rows:
    events = json.loads(row['Events'])
    selected = json.loads(row['CLIP_SelectedFrames'])
    matrix = json.loads(row['CLIP_LogitMatrix'])
    assert len(selected) == len(events) == len(matrix)
    assert all(len(values) == 4 for values in matrix)
    expected_selected = [max(range(4), key=values.__getitem__) + 1 for values in matrix]
    assert selected == expected_selected
    all_scores.extend(score for values in matrix for score in values)
    selection_count += len(selected)

metadata = json.loads(METADATA_JSON.read_text(encoding='utf-8'))
assert metadata['status'] == 'complete'
assert metadata['model']['learned_logit_scale_applied'] is False
assert metadata['similarity']['temperature_or_logit_scale_applied'] is False
assert metadata['similarity']['softmax_applied'] is False
assert min(all_scores) >= -1.0001 and max(all_scores) <= 1.0001

print('validated rows:', len(result_rows))
print('validated event selections:', selection_count)
print('raw cosine range:', min(all_scores), 'to', max(all_scores))
print('selected-frame histogram:', metadata['selected_frame_histogram'])
print('truncated CLIP events:', metadata['text_context']['truncated_event_count'])
print('updated CSV:', EVENTS_CSV)

preview = []
for row in result_rows[:12]:
    preview.append({
        'Id': row['Id'],
        'Events': '\n'.join(json.loads(row['Events'])),
        'SelectedFrames': json.loads(row['CLIP_SelectedFrames']),
        'LogitMatrix': [[round(v, 4) for v in values] for values in json.loads(row['CLIP_LogitMatrix'])],
    })
display(pd.DataFrame(preview))

validated rows: 900
validated event selections: 2033
raw cosine range: 0.0013866610825061798 to 0.38014307618141174
selected-frame histogram: {'1': 517, '2': 495, '3': 502, '4': 519}
truncated CLIP events: 0
updated CSV: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900/events.csv


,Id,Events,SelectedFrames,LogitMatrix
0,Qrt4Ax,A person adjusts a bicycle as it shifts right ...,[4],"[[0.225, 0.26, 0.2699, 0.2798]]"
1,wQMMKq,"The child shows a small object close-up, then ...","[2, 3]","[[0.234, 0.2569, 0.1806, 0.1905], [0.1329, 0.1..."
2,g7LApw,The person stretches their styled hair downwar...,"[2, 1]","[[0.2208, 0.2353, 0.2153, 0.2152], [0.2625, 0...."
3,Z6smZn,"The bumper cars converge in the center, collid...","[1, 3]","[[0.2958, 0.2749, 0.2826, 0.2859], [0.2529, 0...."
4,qGB7Q1,The player in green moves forward and passes t...,"[4, 4, 2]","[[0.2241, 0.041, 0.209, 0.2383], [0.148, 0.096..."
5,xPRZhf,The camera zooms in on a table being painted g...,"[4, 3]","[[0.2874, 0.2338, 0.2214, 0.3259], [0.1828, 0...."
6,wMDNGL,"A wrench tightens a bolt on the bike wheel, th...","[4, 4]","[[0.2445, 0.2623, 0.2524, 0.3123], [0.202, 0.2..."
7,R03j1I,"The woman brushes the horse's side and mane, w...","[4, 2]","[[0.2936, 0.2657, 0.3024, 0.3104], [0.1738, 0...."
8,VCDLdh,The shaker shifts left as the finished cocktai...,"[1, 1]","[[0.1928, 0.1784, 0.1698, 0.1596], [0.2359, 0...."
9,fMfdLR,"The man moves closer to the mirror, tilting hi...","[4, 2]","[[0.2881, 0.1908, 0.183, 0.2972], [0.1962, 0.2..."


## Next steps

The enriched file remains at:

`/content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_900/events.csv`

Interpret `CLIP_SelectedFrames` as a per-event retrieval diagnostic, not yet as a four-frame ordering prediction. Because selections are independent, duplicate frames are expected. The next audit should compare selections with answer-derived temporal frame order and inspect whether truncated or poorly split events fail more often.